# pclass와 age 그리고 gender를 기준으로 전처리

- Surived:0=사망, 1=생존
- Pclass: 1=1등석, 2=2등석, 3=3등석
- gender:male=남성, female=여성
- Age: 나이
- SibSp: 타이타닉 호에 동승한 자매/배우자의 수
- Parch: 타이타닉 호에 동승한 부모/자식의 수
- Ticket: 티켓 번호 X
- Fare: 승객 요금 X
- Embarked: 탑승지; C=셰르부르, Q=퀴즈타운, S=사우샘프턴 => 상관관계 X

In [16]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# train data
df_train = pd.read_csv("./train.csv", encoding="utf-8")

# test data
df_test = pd.read_csv("./test.csv", encoding="utf-8")

df_train.info(), df_train.shape

<class 'pandas.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  916 non-null    int64  
 1   survived     916 non-null    int64  
 2   pclass       916 non-null    int64  
 3   name         916 non-null    str    
 4   gender       916 non-null    str    
 5   age          736 non-null    float64
 6   sibsp        916 non-null    int64  
 7   parch        916 non-null    int64  
 8   ticket       916 non-null    str    
 9   fare         916 non-null    float64
 10  cabin        198 non-null    str    
 11  embarked     915 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 86.0 KB


(None, (916, 12))

In [17]:
df_test.info(), df_test.shape   

<class 'pandas.DataFrame'>
RangeIndex: 393 entries, 0 to 392
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  393 non-null    int64  
 1   pclass       393 non-null    int64  
 2   name         393 non-null    str    
 3   gender       393 non-null    str    
 4   age          310 non-null    float64
 5   sibsp        393 non-null    int64  
 6   parch        393 non-null    int64  
 7   ticket       393 non-null    str    
 8   fare         392 non-null    float64
 9   cabin        97 non-null     str    
 10  embarked     392 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 33.9 KB


(None, (393, 11))

In [ ]:
# -------------------------------------------------
# 시드 고정
# -------------------------------------------------
def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# -------------------------------------------------
# 1. name 기반 feature 생성
# -------------------------------------------------
def add_name_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # family_name 추출
    df["family_name"] = df["name"].str.split(",").str[0].str.strip()

    # title 추출
    df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()

    # title 정리
    title_map = {
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
        "Lady": "Rare",
        "Countess": "Rare",
        "Capt": "Rare",
        "Col": "Rare",
        "Don": "Rare",
        "Dr": "Rare",
        "Major": "Rare",
        "Rev": "Rare",
        "Sir": "Rare",
        "Jonkheer": "Rare",
        "Dona": "Rare",
    }
    df["title"] = df["title"].replace(title_map)

    # 가족 관련 파생변수
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    df["is_alone"] = (df["family_size"] == 1).astype(int)
    df["family_name_count"] = df.groupby("family_name")["family_name"].transform("count")

    return df

# -------------------------------------------------
# 2. train 기준 age reference 생성
# -------------------------------------------------
def fit_age_reference(train_df: pd.DataFrame):
    train_df = train_df.copy()

    age_med_1 = (
        train_df.groupby(["pclass", "gender", "title"])["age"]
        .median()
        .to_dict()
    )

    age_med_2 = (
        train_df.groupby(["pclass", "gender"])["age"]
        .median()
        .to_dict()
    )

    age_med_3 = (
        train_df.groupby(["title"])["age"]
        .median()
        .to_dict()
    )

    age_med_4 = train_df["age"].median()

    return {
        "age_med_1": age_med_1,
        "age_med_2": age_med_2,
        "age_med_3": age_med_3,
        "age_med_4": age_med_4
    }

# -------------------------------------------------
# 3. age 결측치 채우기
# -------------------------------------------------
def fill_age_from_reference(df: pd.DataFrame, ref: dict) -> pd.DataFrame:
    df = df.copy()

    def impute_age(row):
        if pd.notnull(row["age"]):
            return row["age"]

        key1 = (row["pclass"], row["gender"], row["title"])
        if key1 in ref["age_med_1"] and pd.notnull(ref["age_med_1"][key1]):
            return ref["age_med_1"][key1]

        key2 = (row["pclass"], row["gender"])
        if key2 in ref["age_med_2"] and pd.notnull(ref["age_med_2"][key2]):
            return ref["age_med_2"][key2]

        key3 = row["title"]
        if key3 in ref["age_med_3"] and pd.notnull(ref["age_med_3"][key3]):
            return ref["age_med_3"][key3]

        return ref["age_med_4"]

    df["age"] = df.apply(impute_age, axis=1)
    return df

# -------------------------------------------------
# 4. 전체 전처리
# -------------------------------------------------
def preprocess_titanic(train: pd.DataFrame, test: pd.DataFrame, ori_test: pd.DataFrame):
    train = train.copy()
    test = test.copy()
    ori_test = ori_test.copy()

    # name feature 생성
    train = add_name_features(train)
    test = add_name_features(test)
    ori_test = add_name_features(ori_test)

    # age 기준은 train으로만 생성
    age_ref = fit_age_reference(train)

    train = fill_age_from_reference(train, age_ref)
    test = fill_age_from_reference(test, age_ref)
    ori_test = fill_age_from_reference(ori_test, age_ref)

    # embarked, fare 같은 대표 결측치 처리
    if "embarked" in train.columns:
        embarked_mode = train["embarked"].mode(dropna=True)[0]
        train["embarked"] = train["embarked"].fillna(embarked_mode)
        test["embarked"] = test["embarked"].fillna(embarked_mode)
        ori_test["embarked"] = ori_test["embarked"].fillna(embarked_mode)

    if "fare" in train.columns:
        fare_median = train["fare"].median()
        train["fare"] = train["fare"].fillna(fare_median)
        test["fare"] = test["fare"].fillna(fare_median)
        ori_test["fare"] = ori_test["fare"].fillna(fare_median)

    # 불필요 컬럼 삭제
    drop_cols = ["name", "ticket", "cabin", "family_name"]
    existing_drop_cols = [col for col in drop_cols if col in train.columns]

    train = train.drop(columns=existing_drop_cols)
    test = test.drop(columns=existing_drop_cols)
    ori_test = ori_test.drop(columns=existing_drop_cols)

    return train, test, ori_test

train age null: 0
test age null : 0
                                                name family_name title  \
0                     Wheeler, Mr. Edwin Frederick""     Wheeler    Mr   
1                                 Henry, Miss. Delia       Henry  Miss   
2  Hays, Mrs. Charles Melville (Clara Jennings Gr...        Hays   Mrs   
3       Andersson, Mr. August Edvard ("Wennerstrom")   Andersson    Mr   
4                                  Hold, Mr. Stephen        Hold    Mr   

   family_size  is_alone    age  
0            1         1  30.00  
1            1         1  18.25  
2            3         0  52.00  
3            1         1  27.00  
4            2         0  44.00  
                                         name family_name title  family_size  \
0                 McGowan, Miss. Anna "Annie"     McGowan  Miss            1   
1                         Pinsky, Mrs. (Rosa)      Pinsky   Mrs            1   
2           McCarthy, Miss. Catherine Katie""    McCarthy  Miss            

In [27]:
train.head()

,passengerid,survived,pclass,name,gender,age,sibsp,parch,ticket,fare,cabin,embarked,family_name,title,family_size,is_alone,family_name_count
0,0,0,2,"Wheeler, Mr. Edwin Frederick""""",male,30.00,0,0,SC/PARIS 2159,12.8750,NaN,S,Wheeler,Mr,1,1,1
1,1,0,3,"Henry, Miss. Delia",female,18.25,0,0,382649,7.7500,NaN,Q,Henry,Miss,1,1,1
2,2,1,1,"Hays, Mrs. Charles Melville (Clara Jennings Gr...",female,52.00,1,1,12749,93.5000,B69,S,Hays,Mrs,3,0,2
3,3,1,3,"Andersson, Mr. August Edvard (""Wennerstrom"")",male,27.00,0,0,350043,7.7958,NaN,S,Andersson,Mr,1,1,7
4,4,0,2,"Hold, Mr. Stephen",male,44.00,1,0,26707,26.0000,NaN,S,Hold,Mr,2,0,2


In [ ]:
# -------------------------------------------------
# 시드 고정
# -------------------------------------------------
def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# -------------------------------------------------
# 1. name 기반 feature 생성
# -------------------------------------------------
def add_name_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # family_name 추출
    df["family_name"] = df["name"].str.split(",").str[0].str.strip()

    # title 추출
    df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()

    # title 정리
    title_map = {
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
        "Lady": "Rare",
        "Countess": "Rare",
        "Capt": "Rare",
        "Col": "Rare",
        "Don": "Rare",
        "Dr": "Rare",
        "Major": "Rare",
        "Rev": "Rare",
        "Sir": "Rare",
        "Jonkheer": "Rare",
        "Dona": "Rare",
    }
    df["title"] = df["title"].replace(title_map)

    # 가족 관련 파생변수
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    df["is_alone"] = (df["family_size"] == 1).astype(int)
    df["family_name_count"] = df.groupby("family_name")["family_name"].transform("count")

    return df

train = add_name_features(train)
test = add_name_features(test)
ori_te = add_name_features(ori_te)

# -------------------------------------------------
# 2. train 기준 age reference 생성
# -------------------------------------------------
def fit_age_reference(train_df: pd.DataFrame):
    train_df = train_df.copy()

    age_med_1 = (
        train_df.groupby(["pclass", "gender", "title"])["age"]
        .median()
        .to_dict()
    )

    age_med_2 = (
        train_df.groupby(["pclass", "gender"])["age"]
        .median()
        .to_dict()
    )

    age_med_3 = (
        train_df.groupby(["title"])["age"]
        .median()
        .to_dict()
    )

    age_med_4 = train_df["age"].median()

    return {
        "age_med_1": age_med_1,
        "age_med_2": age_med_2,
        "age_med_3": age_med_3,
        "age_med_4": age_med_4
    }
  
train_dict =  fit_age_reference(train)
test_dict =  fit_age_reference(test)
ori_te_dict =  fit_age_reference(ori_te)

# -------------------------------------------------
# 3. age 결측치 채우기
# -------------------------------------------------
def fill_age_from_reference(df: pd.DataFrame, ref: dict) -> pd.DataFrame:
    df = df.copy()

    def impute_age(row):
        if pd.notnull(row["age"]):
            return row["age"]

        key1 = (row["pclass"], row["gender"], row["title"])
        if key1 in ref["age_med_1"] and pd.notnull(ref["age_med_1"][key1]):
            return ref["age_med_1"][key1]

        key2 = (row["pclass"], row["gender"])
        if key2 in ref["age_med_2"] and pd.notnull(ref["age_med_2"][key2]):
            return ref["age_med_2"][key2]

        key3 = row["title"]
        if key3 in ref["age_med_3"] and pd.notnull(ref["age_med_3"][key3]):
            return ref["age_med_3"][key3]

        return ref["age_med_4"]

    df["age"] = df.apply(impute_age, axis=1)
    return df

train = fill_age_from_reference(train, train_dict)
test = fill_age_from_reference(test, test_dict)
ori_te = fill_age_from_reference(ori_te, ori_te_dict)

# -------------------------------------------------
# 4. 전체 전처리
# -------------------------------------------------
def preprocess_titanic(train: pd.DataFrame, test: pd.DataFrame, ori_test: pd.DataFrame):
    train = train.copy()
    test = test.copy()
    ori_test = ori_test.copy()

    # name feature 생성
    train = add_name_features(train)
    test = add_name_features(test)
    ori_test = add_name_features(ori_test)

    # age 기준은 train으로만 생성
    age_ref = fit_age_reference(train)

    train = fill_age_from_reference(train, age_ref)
    test = fill_age_from_reference(test, age_ref)
    ori_test = fill_age_from_reference(ori_test, age_ref)

    # embarked, fare 같은 대표 결측치 처리
    if "embarked" in train.columns:
        embarked_mode = train["embarked"].mode(dropna=True)[0]
        train["embarked"] = train["embarked"].fillna(embarked_mode)
        test["embarked"] = test["embarked"].fillna(embarked_mode)
        ori_test["embarked"] = ori_test["embarked"].fillna(embarked_mode)

    if "fare" in train.columns:
        fare_median = train["fare"].median()
        train["fare"] = train["fare"].fillna(fare_median)
        test["fare"] = test["fare"].fillna(fare_median)
        ori_test["fare"] = ori_test["fare"].fillna(fare_median)

    # 불필요 컬럼 삭제
    drop_cols = ["name", "ticket", "cabin", "family_name","embarked","fare"]
    existing_drop_cols = [col for col in drop_cols if col in train.columns]

    train = train.drop(columns=existing_drop_cols)
    test = test.drop(columns=existing_drop_cols)
    ori_test = ori_test.drop(columns=existing_drop_cols)

    return train, test, ori_test

train, test, ori_test = preprocess_titanic(train, test, ori_test)
